In [22]:
##### Cleans Argentina capital stock data
# reformats

import os
import pandas as pd

In [23]:
##### Load data

# Get the current working directory
cd = os.path.dirname(os.getcwd())

# Import data
capital_stock = pd.ExcelFile(f"{cd}/Data/Raw/Sub_National/Argentina/partido_tractors.xlsx")

ARG_codes = pd.read_csv(f"{cd}/Data/Correspondence_tables/ARG_departments.csv")

# Set save path
save_path = f"{cd}/Data/Clean/Capital_stock/ARG_capital_stock.csv"

In [24]:
### Pull together capital stock data from different tabs

# Get tab names with data
tabs = [s for s in capital_stock.sheet_names if s not in ['Índice', '6.4']]

# Import data for each region
dfs = []
for tab in tabs:
    df = pd.read_excel(
        capital_stock,
        sheet_name=tab,
        header=None,
        skiprows=10,          # skip rows 1–10, so row 11 becomes row 0
        usecols='B,D'         # columns B and D only
    )
    df.columns = ['department', 'tractor_count']
    dfs.append(df)

# Concadinate all regions together 
capital_stock_all = pd.concat(dfs, ignore_index=True)

# Drop any rows where both cols are NaN (blank rows between sections, footers, etc.)
capital_stock_all = capital_stock_all.dropna(subset=['department', 'tractor_count'], how='all')

# convert ot number 
capital_stock_all["tractor_count"] = pd.to_numeric(
    capital_stock_all["tractor_count"].replace("-", 0),
    errors="coerce"
)

/var/folders/48/ky2jtbmj31bfj15cr5gq480w0000gn/T/ipykernel_55644/3986697222.py:27: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  capital_stock_all["tractor_count"].replace("-", 0),


In [25]:
##### Clean capital stock

# Assign occurrence number to each department in capital_stock_all
capital_stock_all['dept_occurrence'] = capital_stock_all.groupby('department').cumcount()

# Assign occurrence number to each department in ARG_codes
ARG_codes['dept_occurrence'] = ARG_codes.groupby('department').cumcount()

# Merge on both department name AND occurrence number
capital_stock_merged = capital_stock_all.merge(
    ARG_codes,
    on=['department', 'dept_occurrence'],
    how='inner'
).drop(columns='dept_occurrence')

# rename columns
capital_stock_merged.rename(columns={'GID_2': 'GEO_ID'}, inplace=True)

capital_stock_merged.rename(columns={'tractor_count': 'ag_capital_stock_count_machinery'}, inplace=True)

# add info
capital_stock_merged['ISO3'] = 'ARG'
capital_stock_merged['Year'] = 2018
capital_stock_merged['GEO_ID_NAME'] = 'GID_2'


# select columns
col_to_keep = ['ISO3', 'GEO_ID', 'GEO_ID_NAME', 'Year', 'ag_capital_stock_count_machinery']
capital_stock_merged = capital_stock_merged[col_to_keep]
capital_stock_merged['ag_capital_stock_count_machinery'] = capital_stock_merged['ag_capital_stock_count_machinery'].fillna(0)

In [26]:
##### Save cleaned data
capital_stock_merged.to_csv(save_path, index=False)